# 03 — Eksperimen model

Training dijalankan lewat CLI/modul yang sama (`wallacea_ml.train`), bukan kode notebook. Notebook ini:
menampilkan konfigurasi & pemeriksaan awal, (opsional) menjalankan training, lalu membaca kurva & model card.

Bobot awal harus berkas LOKAL (lisensi Ultralytics AGPL-3.0/Enterprise). `best.pt` dipilih Ultralytics
berdasarkan split **val**; angka evaluasi akhir hanya dari notebook 04 pada split **test**.

In [ ]:
%matplotlib inline
import json
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
from IPython.display import display

import wallacea_ml
from wallacea_ml.prepare import ML_ROOT
print("wallacea_ml", wallacea_ml.__version__, "| ML_ROOT =", ML_ROOT)


def _df(rows):
    """Tabel: pandas bila terpasang (opsional), selain itu list apa adanya."""
    try:
        import pandas as pd
        return pd.DataFrame(rows)
    except ImportError:
        return rows


In [ ]:
import yaml
from wallacea_ml.train import load_train_config, preflight
DATASET = "uavmulticrop_maize"
MANIFEST = ML_ROOT / "data" / "prepared" / DATASET / "manifest.json"
SPLIT_DIR = ML_ROOT / "data" / "splits" / DATASET
cfg = load_train_config(ML_ROOT / "configs/train/baseline_yolo.yaml", name=f"{DATASET}_yolo11n")
print(yaml.safe_dump(cfg, sort_keys=False))
problems = preflight(MANIFEST, SPLIT_DIR, ML_ROOT / cfg["weights"])
print("pemeriksaan awal:", problems or "lulus")

## Training (opsional dari notebook — lama; lebih aman lewat CLI di terminal)

In [ ]:
RUN_TRAINING = False   # ubah ke True untuk menjalankan; sama dengan:
# .venv\Scripts\python -m wallacea_ml train data\prepared\<DATASET>\manifest.json --split-dir data\splits\<DATASET> --name <nama>
if RUN_TRAINING:
    from wallacea_ml.train import train
    assert train(cfg, MANIFEST, SPLIT_DIR) == 0

## Kurva training (dari run Ultralytics)

In [ ]:
import csv
card_path = ML_ROOT / "artifacts" / "models" / cfg["name"] / "model_card.json"
if not card_path.is_file():
    print("Belum ada model card untuk", cfg["name"], "— training belum dijalankan.")
else:
    card = json.loads(card_path.read_text(encoding="utf-8"))
    rows = list(csv.DictReader(open(Path(card["ultralytics_run_dir"]) / "results.csv", encoding="utf-8")))
    rows = [{k.strip(): float(v) for k, v in r.items()} for r in rows]
    ep = [r["epoch"] for r in rows]
    fig, ax = plt.subplots(1, 2, figsize=(13, 3.5))
    for k in ("train/box_loss", "val/box_loss"): ax[0].plot(ep, [r[k] for r in rows], label=k)
    for k in ("metrics/precision(B)", "metrics/recall(B)", "metrics/mAP50(B)"): ax[1].plot(ep, [r[k] for r in rows], label=k)
    ax[0].legend(); ax[1].legend(); ax[1].set_title("metrik split VAL (dipakai memilih best.pt)"); plt.show()

## Model card (metadata bobot)

In [ ]:
if card_path.is_file():
    card = json.loads(card_path.read_text(encoding="utf-8"))
    print(json.dumps({k: card[k] for k in ("model_id", "model_version", "classes", "api_env", "preprocessing",
                                          "base_weights", "environment", "ultralytics_metrics_val")}, indent=1))
    print("split:", card["dataset"]["counts"], "| group_by:", card["dataset"]["split_report"]["group_by"])